# CN7 가스: 놓친 사례의 관계 이상 지속성
다른 작업의 165번을 기준으로 새 피처를 비교한다. 원본 작업은 수정하지 않는다.
CN7 공식 605샷, 동일 시간블록 및 purge10. 중첩 검증을 유지하되 이미 본 평가자료이므로 개발 성적이다.
입력은 다른 작업에 저장된 163번 원시 샷 CSV/165번 예측 CSV를 읽는다. 실행 전 .venv 활성화.


In [1]:
from pathlib import Path
import json, hashlib, warnings
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import confusion_matrix, average_precision_score
warnings.filterwarnings("ignore", category=RuntimeWarning)
ROOT=Path.cwd()
OUT=ROOT/"reports/tables"
OUT.mkdir(parents=True,exist_ok=True)
SOURCE=OUT/"cn7_reference_snapshot"
if not SOURCE.exists(): SOURCE=Path(r"C:/Users/hayeon/.codex/worktrees/b365/KAMP-IMM/reports/tables")
INPUT=SOURCE/"cn7_163_gas_supervised_input_raw_shots.csv"
d=pd.read_csv(INPUT,parse_dates=["TimeStamp"]).sort_values(["run_id","TimeStamp"]).reset_index(drop=True)
assert len(d)==606 and d.gas.sum()==13
y=d.gas.to_numpy(int)
eligible=(d.gas|d.any_fail.eq(0)).to_numpy()
g=d.run_id.to_numpy()
seq=d.seq.to_numpy()
blocks=d.time_block.to_numpy()
sizes=d.groupby("run_id").TimeStamp.transform("size")
innerblocks=(d.run_id+"__I"+(d.seq*5//sizes).astype(int).astype(str).str.zfill(2)).to_numpy()
N=len(d)
PURGE=10
RELATIONS=[
 ("Max_Injection_Pressure",["Max_Injection_Speed","Filling_Time"]),
 ("Plasticizing_Time",["Max_Screw_RPM","Average_Back_Pressure"]),
 ("Cushion_Position",["Plasticizing_Position","Max_Injection_Pressure"]),
 ("Mold_Temperature_4",["Mold_Temperature_3","Barrel_Temperature_5"])]
VIEW_NAMES=["relation_signed4","relation_abs4","persistent_abs4","relation_plus_persistence8"]
HEADS=["LR","RF","ET","mean3"]
audit=[]; memo={}
def train_context(parent,q):
 ctx=parent.copy()
 for run in pd.unique(g[q]):
  ids=q[g[q]==run]
  ctx[(g==run)&(seq>=seq[ids].min()-PURGE)&(seq<=seq[ids].max()+PURGE)]=False
 return ctx
def residuals(ref):
 normal=ref[y[ref]==0]
 result=[]
 for target,pred in RELATIONS:
  names=[v+"_raw" for v in pred+[target]]
  a=d[names].to_numpy(float)
  imp=SimpleImputer(keep_empty_features=True).fit(a[normal])
  a=imp.transform(a)
  sc=RobustScaler().fit(a[normal,:-1])
  reg=Ridge(alpha=10).fit(sc.transform(a[normal,:-1]),a[normal,-1])
  err=a[:,-1]-reg.predict(sc.transform(a[:,:-1]))
  scale=max(float(np.std(err[normal])),float(np.std(a[normal,-1]))*.1,1e-6)
  result.append(err/scale)
 return np.clip(np.column_stack(result),-20,20)
def views(z,ctx,ids):
 idx=np.flatnonzero(ctx)
 frame=pd.DataFrame(np.abs(z[idx]),index=idx)
 # Removed evaluation records break the history; do not stitch across a purge.
 segment=np.cumsum(np.r_[True,(g[idx][1:]!=g[idx][:-1])|(np.diff(idx)!=1)])
 persistent=frame.groupby(segment,group_keys=False).apply(
  lambda a:a.shift(1).rolling(5,min_periods=3).mean())
 persistent=persistent.reindex(ids).to_numpy(float)
 return {"relation_signed4":z[ids],"relation_abs4":np.abs(z[ids]),
         "persistent_abs4":persistent,
         "relation_plus_persistence8":np.column_stack([np.abs(z[ids]),persistent])}
def predict(parent,q):
 key=(tuple(np.flatnonzero(parent)),tuple(q))
 if key in memo:return memo[key]
 ctx=train_context(parent,q);ref=np.flatnonzero(ctx&eligible)
 assert not ctx[q].any() and set(ref).isdisjoint(q)
 audit.append({"train":len(ref),"train_gas":int(y[ref].sum()),"query":len(q),
 "query_gas":int(y[q].sum()),"timestamp_overlap":len(set(d.TimeStamp.iloc[ref])&set(d.TimeStamp.iloc[q])),
 "test_raw_in_training_history":int(ctx[q].sum())})
 names=[v+"_"+h for v in VIEW_NAMES for h in HEADS]
 if len(np.unique(y[ref]))<2:
  out={n:np.zeros(len(q)) for n in names};memo[key]=(out,ref,ctx);return memo[key]
 z=residuals(ref)
 tr=views(z,ctx,ref);te=views(z,parent,q)
 out={}
 for v in VIEW_NAMES:
  normal=y[ref]==0
  imp=SimpleImputer(keep_empty_features=True).fit(tr[v][normal])
  a=imp.transform(tr[v]);b=imp.transform(te[v])
  sc=RobustScaler().fit(a[normal])
  a=np.clip(sc.transform(a),-20,20);b=np.clip(sc.transform(b),-20,20)
  models={"LR":LogisticRegression(C=.1,class_weight="balanced",max_iter=2000),
   "RF":RandomForestClassifier(n_estimators=40,max_depth=3,min_samples_leaf=3,class_weight="balanced",random_state=42,n_jobs=2),
   "ET":ExtraTreesClassifier(n_estimators=40,max_depth=3,min_samples_leaf=3,class_weight="balanced",random_state=42,n_jobs=2)}
  for h,m in models.items():out[v+"_"+h]=m.fit(a,y[ref]).predict_proba(b)[:,1]
  out[v+"_mean3"]=np.mean([out[v+"_"+h] for h in models],axis=0)
 memo[key]=(out,ref,ctx)
 return memo[key]
def stats(a,b):
 tn,fp,fn,tp=confusion_matrix(a,b,labels=[0,1]).ravel()
 return {"TP":int(tp),"FP":int(fp),"FN":int(fn),"TN":int(tn),
 "precision":float(tp/(tp+fp)) if tp+fp else 0,
 "recall":float(tp/(tp+fn)) if tp+fn else 0,
 "F1":float(2*tp/(2*tp+fp+fn)) if 2*tp+fp+fn else 0}
records=[];choices=[]
baseline=pd.read_csv(SOURCE/"cn7_165_gas_supervised_complementary_predictions.csv",parse_dates=["TimeStamp"])
baseline=baseline.set_index("TimeStamp").loc[d.TimeStamp]
basepred=baseline.three_specialists_confirmed.to_numpy(int)
for held in pd.unique(blocks):
 q=np.flatnonzero(blocks==held)
 outer,ref,ctx=predict(np.ones(N,bool),q)
 inner={n:np.full(len(ref),np.nan) for n in outer}
 for ib in pd.unique(innerblocks[ref]):
  mask=innerblocks[ref]==ib
  ip,_,_=predict(ctx,ref[mask])
  for n in inner:inner[n][mask]=ip[n]
 options=[]
 for n in outer:
  assert np.isfinite(inner[n]).all()
  for quant in [.95,.97,.99]:
   cut=float(np.nextafter(np.quantile(inner[n][y[ref]==0],quant,method="higher"),np.inf))
   met=stats(y[ref],inner[n]>=cut)
   if n.endswith("_mean3"):options.append((met["F1"],met["TP"],-met["FP"],n,quant,cut))
   for j,i in enumerate(q):
    records.append({"model":n,"rule":f"normal{int(quant*100)}","TimeStamp":d.TimeStamp.iloc[i],
     "heldout_block":held,"actual":int(y[i]),"eligible":int(eligible[i]),
     "score":float(outer[n][j]),"threshold":cut,"prediction":int(outer[n][j]>=cut),
     "train_gas":int(y[ref].sum())})
 best=max(options);_,_,_,n,quant,cut=best
 choices.append({"heldout_block":held,"selected":n,"quantile":quant,"inner_F1":best[0],"train_gas":int(y[ref].sum())})
 for j,i in enumerate(q):
  records.append({"model":"inner_selected_mean3","rule":"inner_selection","TimeStamp":d.TimeStamp.iloc[i],
   "heldout_block":held,"actual":int(y[i]),"eligible":int(eligible[i]),
   "score":float(outer[n][j]),"threshold":cut,"prediction":int(outer[n][j]>=cut),
   "train_gas":int(y[ref].sum())})
 print(f"Finished {len(choices)}/20: {held}",flush=True)
 pd.DataFrame(choices).to_csv(OUT/"cn7_170_persistence_progress.csv",index=False)
pp=pd.DataFrame(records)
metrics=[]
for (name,rule),a in pp.groupby(["model","rule"],sort=False):
 a=a.set_index("TimeStamp").loc[d.TimeStamp]
 pr=a.prediction.to_numpy(int);mask=eligible
 for policy,b in [("standalone",pr),("add_to_165",pr|basepred)]:
  metrics.append({"model":name,"rule":rule,"policy":policy,"N":int(mask.sum()),"gas":13,
    **stats(y[mask],b[mask]),"AP_standalone":average_precision_score(y[mask],a.score.to_numpy()[mask])})
mm=pd.DataFrame(metrics)
pp.to_csv(OUT/"cn7_170_persistence_predictions.csv",index=False,encoding="utf-8-sig")
mm.to_csv(OUT/"cn7_170_persistence_metrics.csv",index=False,encoding="utf-8-sig")
pd.DataFrame(audit).to_csv(OUT/"cn7_170_persistence_training_audit.csv",index=False)
assert pd.DataFrame(audit).timestamp_overlap.eq(0).all()
assert pd.DataFrame(audit).test_raw_in_training_history.eq(0).all()
# Descriptive missed-case audit uses labels ONLY here, after evaluation finishes.
miss=(y==1)&(basepred==0)
rows=[]
for held in pd.unique(blocks):
 q=np.flatnonzero(blocks==held)
 ctx=train_context(np.ones(N,bool),q);ref=np.flatnonzero(ctx&eligible)
 z=residuals(ref);vv=views(z,np.ones(N,bool),q)
 ni=ref[y[ref]==0]
 for j,i in enumerate(q):
  if not miss[i]:continue
  row={"TimeStamp":d.TimeStamp.iloc[i],"heldout_block":held,"train_gas":int(y[ref].sum())}
  for k,(target,_) in enumerate(RELATIONS):
   row[target+"_residual_z"]=float(z[i,k])
   row[target+"_abs_normal_percentile"]=float((np.abs(z[ni,k])<=abs(z[i,k])).mean())
   row[target+"_past5_abs_mean"]=float(vv["persistent_abs4"][j,k])
  rows.append(row)
review=pd.DataFrame(rows)
review.to_csv(OUT/"cn7_170_missed8_relation_review.csv",index=False,encoding="utf-8-sig")
prov={"source_sha256":hashlib.sha256(INPUT.read_bytes()).hexdigest(),
 "input":str(INPUT),"target":"CN7 official normal592/gas13; pure shortshot excluded",
 "validation":"same 20 temporal blocks/purge10, nested 5 blocks; NOT unseen-run or future-only",
 "features":"signed/absolute four normal-Ridge residuals; past5 mean absolute residual; current+past8",
 "selection":"new head and cutoff selected only from inner predictions; fixed candidates also reported",
 "caveat":"features informed by previously viewed evaluation misses; baseline165 already test-informed; no untouched independent-test claim",
 "no_test_training_records":True,"no_time_or_run_id_predictors":True}
(OUT/"cn7_170_persistence_provenance.json").write_text(json.dumps(prov,ensure_ascii=False,indent=2),encoding="utf-8")
base=stats(y[eligible],basepred[eligible])
primary=mm[mm.model.eq("inner_selected_mean3")]
top=mm.sort_values(["F1","FP"],ascending=[False,True]).head(12)
textlines=["# CN7 가스: 놓친 8개와 지속형 관계 이상","",
 "## 기존 후보","",str(base),"",
 "기존165: 가스13 중5 탐지, 정상오탐2. 시험 결과를 보고 만든 개발 후보이며 독립 성능이 아니다.","",
 "## 이번 변경","",
 "기존 순간 잔차에 절대편차와 과거5샷 지속 강도를 추가. 정상 관계는 각 학습 구간의 정상으로만 계산한다.",
 "RF·ET·LR 3모델 평균 앙상블. 같은 시간블록/purge10 유지, 평가 기록은 학습 이력에서 제거.",
 "학습은 평가 구간보다 미래 자료도 포함할 수 있어 실시간 미래검증은 아니다.","",
 "## 안쪽 검증으로 선택한 결과","",primary.to_string(index=False),"",
 "## 고정 후보 탐색 결과(외부 성적을 본 사후 최고; 독립 성능 아님)","",top.to_string(index=False),"",
 "## 기존에서 놓친8개","",review.to_string(index=False),"",
 "정상 대비 백분위는 이상 강도이며 불량 확률이 아니다. 정상과 겹치는 사례는 센서만으로 구분 불가능하다고 단정할 수 없지만, 이 관계 피처에서 분리된다고도 주장할 수 없다."]
(ROOT/"reports/cn7_170_persistence_review.md").write_text("\n".join(textlines),encoding="utf-8")
print("\nBASELINE",base)
print("\nPRIMARY\n",primary.to_string(index=False))
print("\nFIXED CANDIDATE SEARCH\n",top.to_string(index=False))
print("\nMISSED REVIEW\n",review.to_string(index=False))


Finished 1/20: cn7_R1__B00


Finished 2/20: cn7_R1__B01


Finished 3/20: cn7_R1__B02


Finished 4/20: cn7_R1__B03


Finished 5/20: cn7_R1__B04


...

Finished 17/20: cn7_R2__B06


Finished 18/20: cn7_R2__B07


Finished 19/20: cn7_R2__B08


Finished 20/20: cn7_R2__B09



BASELINE {'TP': 5, 'FP': 2, 'FN': 8, 'TN': 590, 'precision': 0.7142857142857143, 'recall': 0.38461538461538464, 'F1': 0.5}

PRIMARY
                model            rule     policy   N  gas  TP  FP  FN  TN  precision   recall  F1  AP_standalone
inner_selected_mean3 inner_selection standalone 605   13   0   0  13 592   0.000000 0.000000 0.0       0.036512
inner_selected_mean3 inner_selection add_to_165 605   13   5   2   8 590   0.714286 0.384615 0.5       0.036512

FIXED CANDIDATE SEARCH
                  model     rule     policy   N  gas  TP  FP  FN  TN  precision   recall       F1  AP_standalone
    persistent_abs4_ET normal95 add_to_165 605   13   7   4   6 588   0.636364 0.538462 0.583333       0.030621
    persistent_abs4_ET normal97 add_to_165 605   13   7   4   6 588   0.636364 0.538462 0.583333       0.030621
   relation_signed4_ET normal99 add_to_165 605   13   5   2   8 590   0.714286 0.384615 0.500000       0.042827
relation_signed4_mean3 normal97 add_to_165 605   13   5  